In [2]:
########transfer learning
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# -----------------------------------------------------
# STEP 1: Directories (same as your original code)
# -----------------------------------------------------
Angry_dir = os.path.join(r"C:\Users\tejeswari\Desktop\DML\DL\Train\angry")
Happy_dir = os.path.join(r"C:\Users\tejeswari\Desktop\DML\DL\Train\happy")
Neutral_dir = os.path.join(r"C:\Users\tejeswari\Desktop\DML\DL\Train\neutral")
Sad_dir = os.path.join(r"C:\Users\tejeswari\Desktop\DML\DL\Train\sad")
Surprise_dir = os.path.join(r"C:\Users\tejeswari\Desktop\DML\DL\Train\surprise")

train_Angry_names = os.listdir(Angry_dir)
print(train_Angry_names[:5])

train_Happy_names = os.listdir(Happy_dir)
print(train_Happy_names[:5])

batch_size = 16


['0.jpg', '1.jpg', '10.jpg', '10002.jpg', '10016.jpg']
['100.jpg', '10000.jpg', '10001.jpg', '10005.jpg', '10035.jpg']


In [4]:
# -----------------------------------------------------
# STEP 2: Data generators
# -----------------------------------------------------
# Pretrained models like MobileNetV2 expect 3-channel (RGB) input and
# their own specific pixel scaling, so we use preprocess_input instead
# of a plain rescale=1./255.
train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

# target_size is increased to 96x96 because MobileNetV2 needs a
# minimum input size larger than 48x48 to work well.
target_size = (96, 96)

train_generator = train_datagen.flow_from_directory(
    r"C:\Users\tejeswari\Desktop\DML\DL\Train",
    target_size=target_size,
    batch_size=batch_size,
    # color_mode is 'rgb' here (not 'grayscale'). Keras will
    # automatically duplicate the single channel into 3 channels,
    # since MobileNetV2 was trained on 3-channel ImageNet images.
    color_mode='rgb',
    classes=['Angry', 'Happy', 'Neutral', 'Sad', 'Surprise'],
    class_mode='categorical'
)



Found 24282 images belonging to 5 classes.


In [5]:
# -----------------------------------------------------
# STEP 3: Load the pretrained base model
# -----------------------------------------------------
base_model = MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,     # remove ImageNet's original 1000-class head
    weights='imagenet'
)

# Freeze the pretrained layers so their weights don't change
# during the first phase of training.
base_model.trainable = False



9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 76s 8us/step


In [6]:
# -----------------------------------------------------
# STEP 4: Build the full model (pretrained base + your own head)
# -----------------------------------------------------
inputs = tf.keras.layers.Input(shape=(96, 96, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)

# 5 output neurons for 5 classes with the softmax activation
outputs = tf.keras.layers.Dense(5, activation='softmax')(x)

model = tf.keras.models.Model(inputs, outputs)
model.summary()

model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['acc']
)



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 96, 96, 3)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_96 (Functional)     │ (None, 3, 3, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │          81,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │             325 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,340,293 (8.93 MB)

 Trainable params: 82,309 (321.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [7]:

# -----------------------------------------------------
# STEP 5: Train the new head (base model stays frozen)
# -----------------------------------------------------
total_sample = train_generator.n
num_epochs = 2

model.fit(
    train_generator,
    steps_per_epoch=int(total_sample / batch_size),
    epochs=num_epochs,
    verbose=1
)

Epoch 1/2
1517/1517 ━━━━━━━━━━━━━━━━━━━━ 66s 42ms/step - acc: 0.4565 - loss: 1.3152
Epoch 2/2
1517/1517 ━━━━━━━━━━━━━━━━━━━━ 0s 13us/step - acc: 0.5625 - loss: 0.9880  


C:\Users\tejeswari\AppData\Roaming\Python\Python313\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
